In [16]:
import cv2
import numpy as np
from mtcnn import MTCNN
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input

# Load your trained model (use whichever final model you decided on)
model = load_model('C:/Users/yugka/OneDrive/Desktop/FaceMaskDetect/best_facemask_effnet.keras')

detector = MTCNN()

label_map = {0: 'No Mask', 1: 'Incorrect Mask', 2: 'Mask'}
color_map = {0: (0, 0, 255), 1: (0, 165, 255), 2: (0, 255, 0)}  # BGR: red, orange, green

In [17]:
def process_frame(frame):
    # MTCNN expects RGB, OpenCV frames are BGR by default
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    faces = detector.detect_faces(rgb_frame)

    for face in faces:
        x, y, w, h = face['box']
        x, y = max(0, x), max(0, y)   # guard against negative coords near frame edges

        face_crop = rgb_frame[y:y+h, x:x+w]
        if face_crop.size == 0:
            continue   # skip invalid crops

        # Match your training preprocessing exactly
        face_resized = cv2.resize(face_crop, (128, 128))
        face_array = face_resized.astype('float32')
        face_array = preprocess_input(face_array)        # EfficientNet's own normalization
        face_array = np.expand_dims(face_array, axis=0)

        pred = model.predict(face_array, verbose=0)
        pred_label = np.argmax(pred)
        confidence = pred[0][pred_label]

        label_text = f"{label_map[pred_label]} ({confidence:.2f})"
        color = color_map[pred_label]

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, label_text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    return frame

In [20]:
def process_frame_debug(frame):
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    faces = detector.detect_faces(rgb_frame)

    for face in faces:
        x, y, w, h = face['box']
        x, y = max(0, x), max(0, y)

        face_crop = rgb_frame[y:y+h, x:x+w]
        if face_crop.size == 0:
            continue

        face_resized = cv2.resize(face_crop, (128, 128))
        face_array = face_resized.astype('float32')
        face_array = preprocess_input(face_array)
        face_array = np.expand_dims(face_array, axis=0)

        pred = model.predict(face_array, verbose=0)
        print("Raw probabilities [No Mask, Incorrect, Mask]:", pred[0])

        pred_label = np.argmax(pred)
        confidence = pred[0][pred_label]

        label_text = f"{label_map[pred_label]} ({confidence:.2f})"
        color = color_map[pred_label]

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, label_text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    return frame

In [23]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    output_frame = process_frame_debug(frame)
    cv2.imshow('Mask Detection', output_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Raw probabilities [No Mask, Incorrect, Mask]: [2.3773439e-04 4.7792778e-03 9.9498296e-01]
Raw probabilities [No Mask, Incorrect, Mask]: [1.5995128e-04 2.2469074e-02 9.7737098e-01]
Raw probabilities [No Mask, Incorrect, Mask]: [2.2538540e-04 1.3700908e-02 9.8607373e-01]
Raw probabilities [No Mask, Incorrect, Mask]: [9.3048456e-04 2.0044742e-02 9.7902471e-01]
Raw probabilities [No Mask, Incorrect, Mask]: [4.3254689e-04 1.6723670e-02 9.8284376e-01]
Raw probabilities [No Mask, Incorrect, Mask]: [0.00134126 0.07748822 0.92117053]
Raw probabilities [No Mask, Incorrect, Mask]: [0.00128286 0.03739437 0.96132284]
Raw probabilities [No Mask, Incorrect, Mask]: [0.00107512 0.04450425 0.95442075]
Raw probabilities [No Mask, Incorrect, Mask]: [0.00150238 0.03930562 0.95919204]
Raw probabilities [No Mask, Incorrect, Mask]: [0.00103672 0.03201201 0.9669513 ]
Raw probabilities [No Mask, Incorrect, Mask]: [4.0709844e-04 2.1094358e-02 9.7849852e-01]
Raw probabilities [No Mask, Incorrect, Mask]: [1.677575

In [24]:
from tensorflow.keras.models import load_model
model = load_model('best_facemask_effnet.keras')
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 128, 128,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 128, 128,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 128, 128,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 129, 129,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 64, 64,    │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 64, 64,    │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 64, 64,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 64, 64,    │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 64, 64,    │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 64, 64,    │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 64, 64,    │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 64, 64,    │        512 │ block1a_se_excit

 Total params: 4,542,638 (17.33 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 328,712 (1.25 MB)